# Data Exloration and pseudo Label Generation

In [ ]:
import pandas as pd
import numpy as np
import re
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('outputs', exist_ok=True)
os.makedirs('models', exist_ok=True)

P2N = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}
N2P = {0: 'Low', 1: 'Medium', 2: 'High', 3: 'Critical'}

## Load data

In [ ]:
df = pd.read_csv('data/support_tickets.csv')
print(df.shape)
df.head()

In [ ]:
df['Priority_Level'].value_counts()

In [ ]:
df['Ticket_Channel'].value_counts()

In [ ]:
# resolution time is already in hours - nice
df['Resolution_Time_Hours'].describe()

## Clean and prep

In [ ]:
def clean(text):
    if not isinstance(text, str):
        return ''


    text = text.lower().strip()
    text = re.sub(r"[^\w\s.,!?'-]", ' ', text)
    return re.sub(r'\s+', ' ', text)

def get_tier(email):
    free = {'gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com',
            'icloud.com', 'aol.com', 'example.com', 'example.org', 'example.net'}
    if not isinstance(email, str):
        return 'consumer'
    domain = email.split('@')[-1].lower()

    
    return 'consumer' if domain in free else 'enterprise'




df['subject_clean'] = df['Ticket_Subject'].apply(clean)
df['desc_clean'] = df['Ticket_Description'].apply(clean)
df['priority_num'] = df['Priority_Level'].map(P2N)
df['res_hours'] = pd.to_numeric(df['Resolution_Time_Hours'], errors='coerce')


df['res_hours'] = df['res_hours'].fillna(df['res_hours'].median())
df['domain_tier'] = df['Customer_Email'].apply(get_tier)
df['channel'] = df['Ticket_Channel'].str.lower().str.strip()


df['text_len'] = df['desc_clean'].apply(lambda x: len(x.split()))


df['category'] = df['Issue_Category'].fillna('unknown')
df['ticket_id'] = df['Ticket_ID'].astype(str) if 'Ticket_ID' in df.columns else df.index.astype(str)

print('done')

## Signal 1 — NLP

Keyword matching + text length + channel.

In [ ]:
CRISIS = [
    'outage', 'down', 'breach', 'data loss', 'not working', 'cannot access',
    'system failure', 'completely broken', 'revenue loss', 'security', 'hack',
    'compromised', 'production down', 'all users', 'crash', 'locked out',
    'data breach', 'fraud', 'emergency', 'halted', 'service unavailable'
]
ESCALATION = ['escalate', 'supervisor', 'manager', 'legal', 'lawsuit',
              'refund', 'cancel subscription', 'chargeback', 'attorney']
LOW_WORDS = ['minor', 'suggestion', 'feedback', 'wondering', 'general inquiry',
             'question', 'how do i', 'where is', 'cosmetic', 'font', 'color',
             'when possible', 'no hurry']




def nlp_score(text, channel, tlen):
    crisis = sum(1 for w in CRISIS if w in text)
    esc = sum(1 for w in ESCALATION if w in text)
    low = sum(1 for w in LOW_WORDS if w in text)



    kw = max(0.0, min(1.0, crisis * 0.30 + esc * 0.25 - low * 0.15))
    lc = min(1.0, tlen / 80.0)
    ch = {'chat': 0.20, 'web form': 0.10, 'email': 0.0}.get(channel, 0.0)

    score = 0.50 * kw + 0.35 * lc + 0.15 * ch



    if score >= 0.55:
        return 3
    elif score >= 0.35:
        return 2
    elif score >= 0.18:
        return 1
    return 0


    

df['nlp_sev'] = df.apply(lambda r: nlp_score(r['desc_clean'], r['channel'], r['text_len']), axis=1)
df['nlp_sev'].value_counts().sort_index()

## Signal 2 — Resolution time anomaly



In [ ]:
res_bounds = {}

def res_signal(df):
    sev = df['priority_num'].copy().astype(int)
    for priority, pnum in P2N.items():
        mask = df['Priority_Level'] == priority
        group = df.loc[mask, 'res_hours']
        p15 = group.quantile(0.15)
        p85 = group.quantile(0.85)
        res_bounds[priority] = (float(p15), float(p85))

        sev.loc[mask & (df['res_hours'] <= p15)] = min(3, pnum + 2)
        sev.loc[mask & (df['res_hours'] >= p85)] = max(0, pnum - 2)
    return sev

df['reg_sev'] = res_signal(df)
df['reg_sev'].value_counts().sort_index()

In [ ]:
# check the anomaly boundaries
for p, (lo, hi) in res_bounds.items():
    print(f"{p}: faster than {lo:.0f}h or slower than {hi:.0f}h is anomalous")

In [ ]:
# save these for predict.py to use
with open('models/res_bounds.pkl', 'wb') as f:
    pickle.dump(res_bounds, f)

## Fuse signals and create mismatch labels

In [ ]:
# signal 2 (resolution time) gets 60% weight since text is synthetic and NLP is weaker
df['inferred_num'] = (0.4 * df['nlp_sev'] + 0.6 * df['reg_sev']).round().clip(0, 3).astype(int)
df['inferred'] = df['inferred_num'].map(N2P)
df['delta'] = df['inferred_num'] - df['priority_num']
df['gap'] = df['delta'].abs()
df['mismatch'] = (df['gap'] >= 1).astype(int)

def get_mtype(row):
    if row['mismatch'] == 0:
        return 'Consistent'
    return 'Hidden Crisis' if row['delta'] > 0 else 'False Alarm'

df['mismatch_type'] = df.apply(get_mtype, axis=1)

print(df['mismatch'].value_counts())
print()
print(df['mismatch_type'].value_counts())

## Ablation — how much does each signal contribute?

In [ ]:
s1 = (df['nlp_sev'] == df['priority_num']).mean()
s2 = (df['reg_sev'] == df['priority_num']).mean()
fused = (df['inferred_num'] == df['priority_num']).mean()
pairwise = (df['nlp_sev'] == df['reg_sev']).mean()

print(f"NLP signal agreement with human labels:        {s1:.3f}")
print(f"Resolution signal agreement with human labels: {s2:.3f}")
print(f"Fused agreement with human labels:             {fused:.3f}")
print(f"Pairwise signal agreement (S1 vs S2):          {pairwise:.3f}")

Resolution time signal is clearly stronger on this dataset. NLP contributes but the synthetic text limits its effectiveness.

In [ ]:
df.to_csv('outputs/pseudo_labels.csv', index=False)
print(f"saved {len(df)} rows to outputs/pseudo_labels.csv")